# Merging of the merged Bookings and Measurements dataset and the Materials dataset in respect to the overlapping timeframes in the column created_at

---

In [1]:
import datetime
import os

import duckdb

In [2]:
# Define paths
base_path_merge = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"
base_path_filtered = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"

merged_file = os.path.join(base_path_merge, "merged_bookings_measurements_cleaned.parquet")
filtered_materials_file = os.path.join(base_path_filtered, "filtered_materials_encoded_time_cut.parquet")
merged_full_file = os.path.join(output_dir, "merged_full.parquet")

TIME_WINDOW_DAYS = 14 # Change time frame if desired

In [3]:
con = duckdb.connect()
bookstate_check = con.execute(f"""
    SELECT book_state, COUNT(*) as count
    FROM '{merged_file}'
    GROUP BY book_state
    ORDER BY count DESC;
""").fetchdf()
print("\nBook states in merged_file BEFORE filtering:")
print(bookstate_check)


Book states in merged_file BEFORE filtering:
   book_state     count
0           0  43718793
1           1    215341


In [4]:
# Connect to DuckDB (in-memory engine, but will spill to disk if needed)


# Get start_time, end_time as before
min_created_at_materials = con.execute(f"SELECT MIN(created_at) FROM '{filtered_materials_file}'").fetchone()[0]
min_created_at_merged = con.execute(f"SELECT MIN(created_at) FROM '{merged_file}'").fetchone()[0]
start_time = max(min_created_at_materials, min_created_at_merged)
end_time = start_time + datetime.timedelta(days=TIME_WINDOW_DAYS)

print(f"Time range filter: {start_time} -> {end_time}")

Time range filter: 2025-03-01 02:21:38.306000+01:00 -> 2025-03-15 02:21:38.306000+01:00


In [5]:
bookstate_filtered = con.execute(f"""
    SELECT book_state, COUNT(*) as count
    FROM '{merged_file}'
    WHERE created_at BETWEEN '{start_time}' AND '{end_time}'
    GROUP BY book_state
    ORDER BY count DESC;
""").fetchdf()
print("\nBook states AFTER filtering by time:")
print(bookstate_filtered)


Book states AFTER filtering by time:
   book_state     count
0           0  10147989
1           1     55987


In [6]:
# Count book_state values in the materials dataset
bookstate_materials = con.execute(f"""
    SELECT book_state, COUNT(*) as count
    FROM '{filtered_materials_file}'
    GROUP BY book_state
    ORDER BY count DESC;
""").fetchdf()

print("\nBook states in materials dataset:")
print(bookstate_materials)


Book states in materials dataset:
   book_state     count
0           0  10901301
1           1       463


In [ ]:
# WRITE RESULT DIRECTLY TO PARQUET (no .df())
query = f"""
COPY (
    SELECT
        merged.measure_step_number,
        merged.measure_value,
        merged.book_state,
        merged.measurement_name_encoded,
        merged.measurement_unit_encoded,
        merged.is_within_limits,
        merged.book_state,
        merged.workstep_number_mes,
        merged.book_stamp,
        merged.part_group,
        merged.serial_number_id,
        merged.station_id,
        materials.component_position,
        materials.component_id,
        materials.panel_position,
        materials.supplier_id,
        materials.mounting_place,
        materials.container_number_freq
    FROM '{merged_file}' merged
    LEFT JOIN '{filtered_materials_file}' materials
      ON merged.serial_number_id = materials.serial_number_id
     AND ABS(DATE_DIFF('seconds', merged.created_at, materials.created_at)) <= 5
     AND materials.created_at BETWEEN '{start_time}' AND '{end_time}'
    WHERE merged.created_at BETWEEN '{start_time}' AND '{end_time}'
) TO '{merged_full_file}' (FORMAT PARQUET, COMPRESSION 'zstd');
"""

print("Executing join + direct parquet write in DuckDB...")
con.execute(query)
print(f"Saved FULL merged dataset to {merged_full_file}")
con.close()

Executing join + direct parquet write in DuckDB...


In [9]:
# Reconnect to DuckDB (or reuse your `con` if still open)
con = duckdb.connect()

# Query distinct book_states and their counts
bookstate_query = f"""
SELECT book_state, COUNT(*) AS count
FROM '{merged_full_file}'
GROUP BY book_state
ORDER BY count DESC;
"""

bookstate_df = con.execute(bookstate_query).fetchdf()
print(bookstate_df)

con.close()

   book_state      count
0           0  269942247


## Load and inspect merged dataset

In [8]:
con = duckdb.connect()

# View first few rows
print(con.execute(f"SELECT * FROM '{merged_full_file}' LIMIT 5").fetchdf())

# Count number of rows
row_count = con.execute(f"SELECT COUNT(*) FROM '{merged_full_file}'").fetchone()[0]
print(f"📊 Row count: {row_count}")

# Show column names and types
print(con.execute(f"DESCRIBE SELECT * FROM '{merged_full_file}'").fetchdf())

# Quick summary stats
print(con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(book_stamp) AS min_time,
        MAX(book_stamp) AS max_time
    FROM '{merged_full_file}'
""").fetchdf())
con.close()

   measure_step_number  measure_value  book_state  measurement_name_encoded  \
0                   36          255.0           0                     81322   
1                   36          256.0           0                     81322   
2                   36          255.0           0                     81322   
3                   36          256.0           0                     81322   
4                   36          256.0           0                     81322   

   measurement_unit_encoded  is_within_limits  book_state_1  \
0                   1056712                 1             0   
1                   1056712                 1             0   
2                   1056712                 1             0   
3                   1056712                 1             0   
4                   1056712                 1             0   

   workstep_number_mes                       book_stamp part_group  \
0                    5 2025-03-04 00:35:11.845000+01:00   7a78616d   
1     

## NULL VALUES CHECK

In [9]:
con = duckdb.connect()

# Get all columns
columns = [col[0] for col in con.execute(f"DESCRIBE SELECT * FROM '{merged_full_file}'").fetchall()]
print("Columns in parquet:")
print(columns)

# Check null counts per column
print("\nNull values per column:")
for col in columns:
    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT({col}) AS non_nulls,
        COUNT(*) - COUNT({col}) AS nulls
    FROM '{merged_full_file}'
    """
    result = con.execute(query).fetchdf()
    print(f"{col}: nulls = {result['nulls'][0]} / {result['total_rows'][0]} total")

con.close()

Columns in parquet:
['measure_step_number', 'measure_value', 'book_state', 'measurement_name_encoded', 'measurement_unit_encoded', 'is_within_limits', 'book_state_1', 'workstep_number_mes', 'book_stamp', 'part_group', 'serial_number_id', 'station_id', 'component_position', 'component_id', 'panel_position', 'supplier_id', 'mounting_place', 'container_number_freq']

Null values per column:
measure_step_number: nulls = 0 / 269942247 total
measure_value: nulls = 0 / 269942247 total
book_state: nulls = 0 / 269942247 total
measurement_name_encoded: nulls = 0 / 269942247 total
measurement_unit_encoded: nulls = 0 / 269942247 total
is_within_limits: nulls = 0 / 269942247 total
book_state_1: nulls = 0 / 269942247 total
workstep_number_mes: nulls = 0 / 269942247 total
book_stamp: nulls = 0 / 269942247 total
part_group: nulls = 0 / 269942247 total
serial_number_id: nulls = 0 / 269942247 total
station_id: nulls = 0 / 269942247 total
component_position: nulls = 0 / 269942247 total
component_id: null